# Example 12.2.1 / 2: Add Vehicles, Optional Customers and Open Routes

Companion notebook for **12.2.1 Routing Model Variants** in the AMPL lab Documentation, section "Add Vehicles, Optional Customers and Open Routes". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Add Vehicles, Optional Customers and Open Routes

Three variants change only the degree and balance rows of the load-flow model, so one model holds
them as switches:

| Variant | Row change | Parameter |
|---|---|---|
| several vehicles | the depot sends and receives `vehicles` arcs instead of 1 (`DepotOut`, `DepotIn`) | `vehicles` |
| optional customer | `Leave[j]`, `Enter[j]` and the demand in `Deliver[j]` are multiplied by a binary `Visit[j]`, and serving `j` earns `prize[j]` | `optional`, `prize` |
| open route | arcs back into the depot cost nothing, so the route effectively ends at its last customer | `open_route` |

Arcs stay aggregated (one `Use[i,j]` for all vehicles). The load flow still limits each route:
the load on a route's first arc is that route's total demand, so `LinkLoad` caps it at
`capacity`. The flow also connects every visited customer to the depot only because each one has
positive demand; give a zero-demand customer a small artificial demand, or use MTZ rows. The run
rebuilds every route from the arcs that leave the depot:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES ordered;                           # depot first, then customers
set ARCS within {NODES, NODES};              # allowed directed arcs
param cost {ARCS} >= 0;                      # travel cost per arc
param demand {NODES} >= 0;                   # load delivered to each customer
param capacity > 0;                          # capacity of each vehicle
param vehicles integer >= 1 default 1;       # routes leaving the depot
param optional {NODES} binary default 0;     # 1 = customer may be skipped
param prize {NODES} >= 0 default 0;          # value of serving a customer
param open_route binary default 0;           # 1 = return arcs are free

var Use {ARCS} binary;                       # 1 if some route uses arc (i,j)
var Visit {NODES} binary;                    # 1 if a customer is served
var Load {ARCS} >= 0;                        # load carried on each arc

minimize NetCost:                            # travel cost minus prizes collected
    sum {(i,j) in ARCS: j <> first(NODES) or open_route = 0} cost[i,j] * Use[i,j]
  - sum {j in NODES: j <> first(NODES)} prize[j] * Visit[j];

subject to Required {j in NODES: j <> first(NODES) and optional[j] = 0}: Visit[j] = 1;
subject to DepotOut: sum {(first(NODES),j) in ARCS} Use[first(NODES),j] = vehicles;
subject to DepotIn: sum {(i,first(NODES)) in ARCS} Use[i,first(NODES)] = vehicles;
subject to Leave {i in NODES: i <> first(NODES)}: sum {(i,j) in ARCS} Use[i,j] = Visit[i];
subject to Enter {j in NODES: j <> first(NODES)}: sum {(i,j) in ARCS} Use[i,j] = Visit[j];
subject to LinkLoad {(i,j) in ARCS}: Load[i,j] <= capacity * Use[i,j];
subject to Deliver {j in NODES: j <> first(NODES)}:  # each served customer keeps its demand
    sum {(i,j) in ARCS} Load[i,j] - sum {(j,k) in ARCS} Load[j,k] = demand[j] * Visit[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [5]:
# run.py
def routes():
    arcs = [arc for arc, used in ampl.var["Use"].to_dict().items() if used > 0.5]
    found = []
    for first_stop in [j for (i, j) in arcs if i == "depot"]:
        route = ["depot", first_stop]
        while route[-1] != "depot":
            route.append(next(j for (i, j) in arcs if i == route[-1]))
        found.append("-".join(route))
    return found

cases = {
    "one vehicle, capacity 9": {"capacity": 9},
    "two vehicles, capacity 9": {"capacity": 9, "vehicles": 2},
    "c3, c4 optional, prize 6": {"optional": {"c3": 1, "c4": 1}, "prize": {"c3": 6, "c4": 6}},
    "c3, c4 optional, prize 8": {"optional": {"c3": 1, "c4": 1}, "prize": {"c3": 8, "c4": 8}},
    "open route": {"open_route": 1},
}
for label, case in cases.items():
    ampl.param["capacity"] = case.get("capacity", 15)
    ampl.param["vehicles"] = case.get("vehicles", 1)
    ampl.param["open_route"] = case.get("open_route", 0)
    ampl.param["optional"] = {n: case.get("optional", {}).get(n, 0) for n in nodes}
    ampl.param["prize"] = {n: case.get("prize", {}).get(n, 0) for n in nodes}
    ampl.solve(solver="highs", verbose=False)
    if ampl.solve_result == "solved":
        print(f"{label}: {ampl.obj['NetCost'].value():g}", routes())
    else:
        print(f"{label}: {ampl.solve_result}")

one vehicle, capacity 9: infeasible
two vehicles, capacity 9: 24 ['depot-c1-c2-depot', 'depot-c3-c4-depot']
c3, c4 optional, prize 6: 5 ['depot-c1-c2-depot']
c3, c4 optional, prize 8: 4 ['depot-c1-c3-c4-c2-depot']
open route: 12 ['depot-c1-c2-c4-c3-depot']


### Pure AMPL: model.run

Pure AMPL, with the same model; data.dat holds the `NODES`, `ARCS` and `cost` tables from the
time-window example plus `param demand := depot 0  c1 3  c2 4  c3 2  c4 5;` and
`param capacity := 15;`. `let` changes the switches between solves:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
let capacity := 9;
solve;
printf "one vehicle, capacity 9: %s\n", solve_result;
let vehicles := 2;
solve;
printf "two vehicles, capacity 9: %g\n", NetCost;
let vehicles := 1;
let capacity := 15;
let {j in {"c3", "c4"}} optional[j] := 1;
let {j in {"c3", "c4"}} prize[j] := 6;
solve;
printf "c3, c4 optional, prize 6: %g\n", NetCost;
let {j in {"c3", "c4"}} prize[j] := 8;
solve;
printf "c3, c4 optional, prize 8: %g\n", NetCost;
let {j in NODES} optional[j] := 0;
let {j in NODES} prize[j] := 0;
let open_route := 1;
solve;
printf "open route: %g\n", NetCost;
display {(i,j) in ARCS: Use[i,j] > 0.5};


suffix dunbdd OUT;
one vehicle, capacity 9: infeasible
two vehicles, capacity 9: 24
c3, c4 optional, prize 6: 5
c3, c4 optional, prize 8: 4
open route: 12
set {(i,j) in ARCS: Use[i,j] > 0.5}  :=
(depot,c1)   (c1,c2)      (c2,c4)      (c3,depot)   (c4,c3);

